# 00 — Generate Synthetic Data

Goal: create a synthetic binary-classification dataset that mimics the target experiment scale
(**10–20M rows x 50 features**), with a *known* logistic data-generating process (DGP), so we can
later judge how close each model's predictions/coefficients get to the ground truth.

**Design choices**

- Features `X ~ N(0, 1)`, i.i.d., float32 (halves memory vs float64; at 20M x 50 that's the
  difference between ~4GB and ~8GB for `X` alone).
- True linear index `logit = X @ true_beta + intercept`, plus an unobserved noise term (so the
  Bayes error rate isn't zero — otherwise every model looks perfect).
- Labels drawn as `y ~ Bernoulli(sigmoid(logit + noise))` — this is exactly the model
  **logistic regression** assumes, and only an *approximation* of what a **linear probability
  model (LPM)** assumes (LPM assumes `E[y|X]` is linear in X, not sigmoid-linear) — that
  mismatch is intentional, it's what you'd see with real data too.
- Data is generated and written **chunk by chunk directly into a `.npy` memmap on disk**, so
  peak RAM never exceeds one chunk regardless of the final row count — this is what makes it
  safe to scale `N_ROWS` up to 20,000,000 on a 32GB machine.

Run this notebook once per `N_ROWS` value you want to test; it caches the arrays under `data/`
so `01_...` and `02_...` can just load them.

In [1]:
import os
import time
from pathlib import Path

import numpy as np
import psutil

# ---------------------------------------------------------------------------
# CONFIG — every knob for this notebook lives here. Each can also be set from
# outside the notebook (e.g. on the GCP Workbench, or via `papermill`/
# `jupyter nbconvert --execute`) by exporting the matching env var before
# launching Jupyter, so you don't have to hand-edit the notebook per run:
#   N_ROWS=10000000 N_FEATURES=50 jupyter nbconvert --execute --to notebook \
#       00_generate_synthetic_data.ipynb
#
# Smoke-test defaults are small so this notebook runs in seconds as shipped.
# For the real GCP Workbench experiment, set N_ROWS to 10_000_000 or
# 20_000_000. Rules of thumb for RAM budget (float32 X only, 50 features):
#   10,000,000 rows -> ~2.0 GB for X   (comfortable on 32GB)
#   20,000,000 rows -> ~4.0 GB for X   (comfortable on 32GB, very safe on 64GB)
# sklearn will make additional working copies during fit/split, so budget
# 3-5x the raw array size as a rough safety margin.
# ---------------------------------------------------------------------------
N_ROWS = int(os.environ.get("N_ROWS", 200_000))            # <-- set to 10_000_000 / 20_000_000 for the real run
N_FEATURES = int(os.environ.get("N_FEATURES", 50))
RANDOM_STATE = int(os.environ.get("RANDOM_STATE", 42))
CHUNK_SIZE = int(os.environ.get("CHUNK_SIZE", 1_000_000))   # rows generated per chunk; bounds peak RAM during generation
TRUE_BETA_SCALE = float(os.environ.get("TRUE_BETA_SCALE", 0.5))  # spread of the ground-truth coefficients
NOISE_SCALE = float(os.environ.get("NOISE_SCALE", 1.0))          # unobserved-noise std dev (controls Bayes error)
TRUE_INTERCEPT = float(os.environ.get("TRUE_INTERCEPT", -0.3))
FORCE_REGENERATE = os.environ.get("FORCE_REGENERATE", "0") == "1"  # set to "1" to overwrite cached data

DATA_DIR = Path(os.environ.get("DATA_DIR", "data"))
DATA_DIR.mkdir(exist_ok=True)
X_PATH = DATA_DIR / f"X_{N_ROWS}.npy"
Y_PATH = DATA_DIR / f"y_{N_ROWS}.npy"

print(f"N_ROWS={N_ROWS:,}  N_FEATURES={N_FEATURES}")
print(f"Estimated X size on disk: {N_ROWS * N_FEATURES * 4 / 1e9:.2f} GB (float32)")
print(f"Available system RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")

N_ROWS=200,000  N_FEATURES=50
Estimated X size on disk: 0.04 GB (float32)
Available system RAM: 17.2 GB


In [2]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def generate_dataset(n_rows, n_features, x_path, y_path, chunk_size, random_state,
                      beta_scale, noise_scale, intercept):
    """Writes X (n_rows x n_features, float32) and y (n_rows, float32) directly to
    disk-backed .npy memmaps, one chunk at a time, so peak RAM stays bounded by
    `chunk_size` regardless of `n_rows`. This is what actually generates the data
    used by notebooks 01 and 02 — nothing downstream regenerates it."""
    rng = np.random.default_rng(random_state)

    # Ground-truth coefficients for the logistic DGP. Kept small-ish (beta_scale)
    # so class probabilities aren't saturated near 0/1 for every row.
    true_beta = rng.normal(scale=beta_scale, size=n_features).astype(np.float32)
    true_intercept = np.float32(intercept)

    X_mm = np.lib.format.open_memmap(x_path, mode="w+", dtype=np.float32, shape=(n_rows, n_features))
    y_mm = np.lib.format.open_memmap(y_path, mode="w+", dtype=np.float32, shape=(n_rows,))

    t0 = time.perf_counter()
    for start in range(0, n_rows, chunk_size):
        end = min(start + chunk_size, n_rows)
        n = end - start
        Xc = rng.normal(size=(n, n_features)).astype(np.float32)
        logits = Xc @ true_beta + true_intercept
        noise = rng.normal(scale=noise_scale, size=n).astype(np.float32)  # unobserved noise -> irreducible error
        p = sigmoid(logits + noise)
        yc = (rng.random(n) < p).astype(np.float32)
        X_mm[start:end] = Xc
        y_mm[start:end] = yc
    X_mm.flush()
    y_mm.flush()
    elapsed = time.perf_counter() - t0
    return X_mm, y_mm, true_beta, true_intercept, elapsed


if X_PATH.exists() and Y_PATH.exists() and not FORCE_REGENERATE:
    print(f"Found cached data at {X_PATH} / {Y_PATH}, skipping generation.")
    print("Set FORCE_REGENERATE=1 (or change N_ROWS) to regenerate.")
    X_mm = np.load(X_PATH, mmap_mode="r")
    y_mm = np.load(Y_PATH, mmap_mode="r")
else:
    X_mm, y_mm, true_beta, true_intercept, elapsed = generate_dataset(
        N_ROWS, N_FEATURES, X_PATH, Y_PATH, CHUNK_SIZE, RANDOM_STATE,
        TRUE_BETA_SCALE, NOISE_SCALE, TRUE_INTERCEPT,
    )
    np.save(DATA_DIR / f"true_beta_{N_ROWS}.npy", true_beta)
    print(f"Generated {N_ROWS:,} rows x {N_FEATURES} features in {elapsed:.2f}s")
    print(f"X: {X_PATH}  ({X_PATH.stat().st_size / 1e9:.2f} GB)")
    print(f"y: {Y_PATH}  ({Y_PATH.stat().st_size / 1e9:.2f} GB)")

Found cached data at data/X_200000.npy / data/y_200000.npy, skipping generation.
Set FORCE_REGENERATE=1 (or change N_ROWS) to regenerate.


In [3]:
# Sanity checks
print("X shape:", X_mm.shape, X_mm.dtype)
print("y shape:", y_mm.shape, y_mm.dtype)
print(f"Positive class rate: {y_mm.mean():.3f}")
print(f"X mean/std (should be ~0/~1): {X_mm[:50_000].mean():.3f} / {X_mm[:50_000].std():.3f}")

X shape: (200000, 50) float32
y shape: (200000,) float32
Positive class rate: 0.465
X mean/std (should be ~0/~1): 0.001 / 1.000


## Notes for scaling this up on the GCP Workbench

- **32GB machine**: comfortably handles 10M rows. 20M rows is workable but leaves less headroom
  for the batch `LogisticRegression` solver's internal copies during `train_test_split` — prefer
  `SGDClassifier` (partial_fit / streaming) if you hit `MemoryError`, or drop to float32
  throughout (already done here) and avoid `pandas.DataFrame` wrappers (keep everything as raw
  numpy arrays).
- **64GB machine**: comfortably handles 20M rows with all three models, including the full-batch
  `LogisticRegression(solver="lbfgs")` baseline.
- Re-run this notebook once per scale you want to test (e.g. `N_ROWS = 1_000_000`, then
  `10_000_000`, then `20_000_000`) — each size is cached under `data/` with the row count in the
  filename, so `01_...` and `02_...` just need `N_ROWS` set to match.